# Module 1: Meet Jev: Typed Decisions Instead of Text

*Getting Started with Jev - Data Trainers LLC - Axel Sirota*

## The story

You run the support platform for a personal assistant app. Every day thousands of messages
come in: "i lost my card", "book a table for two", "what's the weather tomorrow". Each one
has to reach the right team, and some of them cannot wait. Today that sorting is done by
hand.

In this course you build the triage layer that does it in code. The engine is **Jev**, a new
kind of model from TypeSafe AI: it answers questions with typed answers your code can use
directly. Our data is CLINC150, a public set of 23,850 real assistant messages across 10
domains and 150 intents.

## What you will be able to do

- Say what Jev is, and how it differs from an LLM
- Call Jev from Python with Choice, Score and Noul questions, and read the probabilities and confidence it returns
- Compare Jev with an LLM and with a trained classifier on the same triage question: what comes back, how fast, at what cost
- Turn Jev's judgments into an action with rules in your own code

## Prerequisites

- Python in notebooks and basic classification metrics (accuracy)
- A Jev API key and an OpenAI API key (given in class)

## How this notebook works

Every Part runs in four beats:

| Beat | What happens |
|---|---|
| 1. The problem | what goes wrong, with a real example |
| 2. The solution | the idea that fixes it |
| 3. Demo | we run it together: first the problem, then the solution |
| 4. Lab | you build it, in three levels: Guided, Core, Challenge (optional) |

Numbers written in the text come from our own run on 2026-10-31. Yours will differ a little:
trust the numbers your cells print.

# Setup

Now let's make it real. Run the next four cells in order: install, keys, config, data. The
key cell asks for each key once; keys stay in this runtime only and are never printed.

If Colab shows a "Restart
session" button after the install, click it, then continue from the KEYS cell.

In [ ]:
%pip install -q typesafe-sdk==0.7.5 openai==3.41.0 datasets==5.1.0

In [ ]:
# ===========================================================================
# KEYS: Jev and OpenAI, typed once, never printed
# ===========================================================================
# Why: keys never live in the notebook. If your environment already has them, we use them.
import os
from getpass import getpass

for key_name in ("TYPESAFE_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(key_name):
        os.environ[key_name] = getpass(f"{key_name}: ")
    if not os.environ[key_name].strip():
        raise ValueError(f"{key_name} is empty. Ask your instructor for the class key.")
print("Both keys are set for this runtime.")

In [ ]:
# ===========================================================================
# CONFIG: clients, pinned models, prices, one probe call
# ===========================================================================
# Why: pinned model ids keep today's numbers reproducible next month.
# Timeouts and retries are set on purpose, not left to defaults. The Jev SDK retries 408, 429
# and 5xx up to 2 times with backoff inside a 30 s budget, then raises
# (docs.typesafe.ai/sdk/python/api/retries.md). The whole room shares one key, so a 429 (rate
# limit, counted per second per organization: docs.typesafe.ai/models.md) is possible. If a
# cell raises, wait a moment and run it again; do not loop it.
import statistics
import time

from openai import OpenAI
from typesafe_sdk import Choice, Noul, RetryPolicy, Score, TypeSafeClient

JEV_MODEL = "jev-1.14.0"                 # pinned; "jev-latest" moves with each release
LLM_MODEL = "gpt-5.4-mini-2026-03-17"    # pinned OpenAI snapshot
LOGPROB_MODEL = "gpt-4.1-mini-2025-04-14"  # older OpenAI model that still returns logprobs

# Prices in USD per token, from docs.typesafe.ai/models.md (Jev: $0.042 per million input
# tokens, output free) and the OpenAI model pages on developers.openai.com.
JEV_PRICE_IN = 0.042 / 1_000_000
LLM_PRICE_IN, LLM_PRICE_OUT = 0.75 / 1_000_000, 4.50 / 1_000_000
LOGPROB_PRICE_IN, LOGPROB_PRICE_OUT = 0.40 / 1_000_000, 1.60 / 1_000_000

jev = TypeSafeClient(retry=RetryPolicy(max_retries=2, timeout=30.0))
llm = OpenAI(timeout=30.0, max_retries=2)

probe = jev.system_one(state={"message": "hello there"}, model=JEV_MODEL,
                       questions={"greeting": Noul(instructions="Is `message` a greeting?")})
assert probe.model == JEV_MODEL, f"expected {JEV_MODEL}, got {probe.model}"
llm.models.retrieve(LLM_MODEL)
print(f"Jev answered with {probe.model}: greeting = {probe.answers['greeting'].noul:.2f}")
print(f"OpenAI model {LLM_MODEL} is available.")

In [ ]:
# ===========================================================================
# DATA: CLINC150 messages and the 10 teams
# ===========================================================================
# Why: CLINC150 labels intents. The authors' domains.json maps each intent to one of 10
# domains; we treat each domain as a team in our support org. The 1,000 out-of-scope
# messages (no team fits) are left out today on purpose: Module 3 is about them.
import json
import random
import urllib.request

from datasets import load_dataset

DOMAINS_URL = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/domains.json"
intent_to_team = {intent: team
                  for team, intents in json.load(urllib.request.urlopen(DOMAINS_URL)).items()
                  for intent in intents}

clinc = load_dataset("clinc/clinc_oos", "plus", split="test")
intent_names = clinc.features["intent"].names
in_scope = [(row["text"], intent_to_team[intent_names[row["intent"]]])
            for row in clinc if intent_names[row["intent"]] != "oos"]

TEAMS = {
    "banking": "accounts, balances, transfers, bills, direct deposit",
    "credit_cards": "credit cards: limits, rewards, lost or damaged cards, APR",
    "kitchen_and_dining": "recipes, cooking, nutrition, restaurants",
    "home": "smart home, shopping and todo lists, reminders, calendar",
    "auto_and_commute": "car maintenance, gas, traffic, rides, directions",
    "travel": "flights, hotels, luggage, visas, travel alerts, translation",
    "utility": "time, date, weather, timers, calculator, definitions",
    "work": "payroll, PTO, taxes, 401k, meetings",
    "small_talk": "greetings, jokes, chit-chat with the assistant",
    "meta": "settings of the assistant itself: name, language, voice, volume",
}

rng = random.Random(42)
sample = []
for team in TEAMS:
    sample += rng.sample([pair for pair in in_scope if pair[1] == team], 4)

print(f"{len(clinc)} test messages, {len(in_scope)} in scope; our sample: {len(sample)} (4 per team)")
for text, team in sample[:3]:
    print(f"  {team:14} | {text}")

## Part 1: Typed answers instead of text

The first piece of our triage layer: given a message, which of our 10 teams should handle it?

### 1. The problem

Ask an LLM, even with a JSON schema of the 10 teams, and you get a label with **no probability**: an easy message and a hard one look the same, so your code cannot tell which one a person should check. Each one-word answer is a full LLM call: 0.7 to 0.9 seconds and $0.18 to $0.26 per 1,000 messages in our run.

**Figure: The problem - an easy and a hard message through a schema LLM: two bare labels, no number, so the hard one is routed to the wrong team. Red is what fails.**

<!-- DIAGRAM: p1-problem-llm-text -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-problem-llm-text.png" width="750" alt="Top row: two messages, the easy lost credit card and flight to Rome, and the hard abort mission, go to an LLM with a JSON schema of the 10 teams, measured at p50 879 ms and 0.256 dollars per 1,000; it answers credit_cards, and small_talk where the true team is meta, with no number for how sure. Bottom row: your code cannot tell easy from hard, so both are routed and the hard one goes to the wrong team, missed">

### 2. The solution

Use a model built for decisions. **Jev is a decision model.** It reads a state, answers your questions with typed values, and never writes text. TypeSafe AI calls it a **System One model**, after Kahneman's fast, at-a-glance judgments ("is this email angry?").

**Figure: What is Jev - follow one message: state and questions go in, one call, typed answers come out, your code decides.**

<!-- DIAGRAM: what-is-jev -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/what-is-jev.png" width="630" alt="You send one call, jev.system_one, with the state, a message about a lost card that must be frozen, and three questions: team as a Choice, urgency as a Score, risk as a Noul. Jev answers team credit_cards with a probability per team, urgency 2.00, risk 0.97; your code checks risk at least 0.7 and the outcome is ESCALATE">

One request can carry many questions: the state is read, and paid for, once. Part 2 measures it.

#### Three question types

Each question has a type, its wording (`instructions`) and, for Choice and Score, its options (`criteria`). A name in backticks, like `` `message` ``, points Jev at that field of the state.

**Figure: Three question types - one message, three questions: what you write and what Jev returns (numbers from our run).**

<!-- DIAGRAM: three-question-types -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/three-question-types.png" width="750" alt="One message about a flight to Rome and a declined card feeds three rows. Choice: team, criteria a dict of options, answer choice credit_cards with credit_cards 0.93 and travel 0.06, confidence 0.92. Score: frustration, criteria a list of levels calm, annoyed, angry, answer score 1.42 with level probabilities 0.08, 0.42, 0.50, confidence 0.13. Noul: travel_plans, no criteria, answer one probability 0.99">

In code, the call from the first figure:

```python
response = jev.system_one(
    state={"message": "i lost my card at the mall and need it frozen right away"},
    questions={
        "team": Choice(instructions="Which team should handle `message`?",
                       criteria={"banking": "accounts and transfers",           # dict: option -> line
                                 "credit_cards": "cards: limits, lost cards"}),
        "urgency": Score(instructions="How urgent is `message`?",
                         criteria=["can wait", "within days", "within the hour"]),  # list: level 0 first
        "risk": Noul(instructions="`message` says money or account access is at risk"),  # no criteria
    },
    model="jev-1.14.0",
)
```

Every response also says which model version answered: `jev-latest` moves with each release, a pinned id like `jev-1.14.0` does not.

#### Back to our problem

Ask Jev a **Choice**: the 10 teams are the options, written as data.

```python
team_question = Choice(instructions="Which team should handle `message`?", criteria=TEAMS)
```

Jev can only answer with one of those options, and it returns a **probability for every option**. "credit_cards 0.99" and "travel 0.22, meta 0.20" (both from our run) now look different, and your code can act on the difference.

**The rule: ask Jev a typed question and you get back data, not text.**

**Figure: The solution - the same two messages and layout, now through Jev: the easy one comes back sure, the hard one torn, so your code routes one and sends the other to review.**

<!-- DIAGRAM: p1-solution-jev-choice -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-solution-jev-choice.png" width="750" alt="Top row: the same two messages go to Jev with team_question, a Choice of the 10 teams, measured at p50 235 ms and 0.022 dollars per 1,000; the easy one gets credit_cards 0.99 and travel 0.01, sure; abort mission gets travel 0.22 and meta 0.20, confidence 0.13, torn. Bottom row: your code reads the number, routes the easy message, and sends the hard one, 0.13 below your line, to review by an agent">

Use an LLM when you need words (a reply to the customer). Use Jev when your code needs a decision.


#### Reading the numbers: score and confidence

A Score is the probability-weighted level, so a middle score can mean "Jev is split", not "medium". Confidence (Choice and Score) runs from 0, spread out, to 1, all on one option. For a Choice with `n` options it is `(top - 1/n) / (1 - 1/n)`: `(0.93 - 0.1) / 0.9 = 0.92`. A Score's confidence measures how far the probability spreads from the most likely level, so the split frustration gets 0.13 (docs.typesafe.ai/confidence).

**Figure: Score and confidence - the score is computed from the level probabilities; confidence says how decided Jev is.**

<!-- DIAGRAM: score-and-confidence -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/score-and-confidence.png" width="730" alt="Top: frustration level probabilities 0.08 calm, 0.42 annoyed, 0.50 angry give score 0 times 0.08 plus 1 times 0.42 plus 2 times 0.50, which is 1.42: split between 1 and 2, not medium, confidence 0.13. Bottom: a Choice of 10 teams; the Rome flight message gets credit_cards 0.93 and travel 0.06, confidence 0.92, decided; abort mission gets travel 0.22 and meta 0.20, confidence 0.13, torn">

**Confidence says how decided Jev is, not whether it is right.** TypeSafe describes the probabilities as calibrated; Module 3 checks that on our data.

### 3. Demo

Same message, several ways: an LLM with a plain prompt, the same LLM with a JSON schema, an
LLM's token log-probabilities, and Jev. Then the baseline any engineer should ask for: a
classifier trained on CLINC150's labeled messages. Finally all five on 40 messages, side by
side. In the table, **p50** is the median latency and **p95** the slow tail.

In [ ]:
# ===========================================================================
# DEMO: the problem: ask an LLM which team should handle a message (NARRATE)
# ===========================================================================
# Why: this is how many teams start: a prompt that lists the options and asks for the name.
# Both LLM arms get the SAME prompt, including "reply with only the team name". The only
# difference is that schema=True also forces a JSON reply whose team must be one of the 10.
TEAM_LIST = "\n".join(f"- {name}: {desc}" for name, desc in TEAMS.items())
TEAM_SCHEMA = {"type": "json_schema", "json_schema": {"name": "triage", "strict": True, "schema": {
    "type": "object", "properties": {"team": {"type": "string", "enum": list(TEAMS)}},
    "required": ["team"], "additionalProperties": False}}}


def team_prompt(message: str) -> str:
    return ("You triage messages for a personal assistant app. Which team should handle "
            f"this message?\nTeams:\n{TEAM_LIST}\n\nMessage: {message}\n\n"
            "Reply with only the team name, exactly as written above.")


def ask_llm(message: str, schema: bool = False) -> dict:
    """Ask gpt-5.4-mini for the team. schema=True forces a JSON reply with one of the teams."""
    extra = {"response_format": TEAM_SCHEMA} if schema else {}
    start = time.perf_counter()
    reply = llm.chat.completions.create(model=LLM_MODEL, reasoning_effort="none",
                                        messages=[{"role": "user", "content": team_prompt(message)}],
                                        **extra)
    ms = (time.perf_counter() - start) * 1000
    text = reply.choices[0].message.content.strip()
    team = json.loads(text)["team"] if schema else (text if text in TEAMS else None)
    usage = reply.usage
    return {"text": text, "team": team, "ms": ms, "probability": None,
            "cost": usage.prompt_tokens * LLM_PRICE_IN + usage.completion_tokens * LLM_PRICE_OUT}


query = "i lost my credit card and i fly to rome tomorrow"
answer = ask_llm(query)
print(f"=== LLM, plain prompt ({answer['ms']:.0f} ms) ===")
print(repr(answer["text"]))
print("exact team name?", answer["team"] is not None)

In [ ]:
# ===========================================================================
# DEMO: the fix everyone reaches for: a JSON schema (NARRATE)
# ===========================================================================
# Why: an enum in the schema forces one of the 10 team names. The format problem is gone.
# What is still missing: any sign of how sure the model was.
answer = ask_llm(query, schema=True)
print(f"=== LLM, JSON schema ({answer['ms']:.0f} ms) ===")
print(repr(answer["text"]), "->", answer["team"])

In [ ]:
# ===========================================================================
# DEMO: can the LLM say how sure it is? Token log-probabilities (NARRATE)
# ===========================================================================
# Why: OpenAI can return logprobs: the probability of each TOKEN it wrote, plus the top
# alternatives for that token. Our pinned model refuses them (a 403 we catch on purpose, it is
# not flaky). An older model still returns them, so we use it to see what they give you.
import math

from openai import PermissionDeniedError

try:
    llm.chat.completions.create(model=LLM_MODEL, reasoning_effort="none", logprobs=True,
                                messages=[{"role": "user", "content": team_prompt(query)}])
except PermissionDeniedError as error:
    print(f"{LLM_MODEL}: logprobs refused ({error.status_code})")
else:
    print(f"{LLM_MODEL}: logprobs returned (OpenAI changed this since we measured)")


def ask_llm_logprobs(message: str) -> dict:
    """Ask LOGPROB_MODEL for the team; probability = product of the reply's token probabilities."""
    start = time.perf_counter()
    reply = llm.chat.completions.create(model=LOGPROB_MODEL, logprobs=True, top_logprobs=5,
                                        messages=[{"role": "user", "content": team_prompt(message)}])
    ms = (time.perf_counter() - start) * 1000
    text = reply.choices[0].message.content.strip()
    tokens = reply.choices[0].logprobs.content
    usage = reply.usage
    return {"text": text, "team": text if text in TEAMS else None, "ms": ms,
            "probability": math.exp(sum(t.logprob for t in tokens)),
            "first_token_top5": [(t.token, math.exp(t.logprob)) for t in tokens[0].top_logprobs],
            "cost": usage.prompt_tokens * LOGPROB_PRICE_IN + usage.completion_tokens * LOGPROB_PRICE_OUT}


answer = ask_llm_logprobs(query)
print(f"\n=== {LOGPROB_MODEL}, logprobs ({answer['ms']:.0f} ms) ===")
print(f"reply {answer['text']!r}, probability of those tokens {answer['probability']:.2f}")
print("first token, top 5:", ", ".join(f"{tok!r} {p:.2f}" for tok, p in answer["first_token_top5"]))
print("These are tokens, not your 10 teams: a team that is not in the top 5 gets no number.")

In [ ]:
# ===========================================================================
# DEMO: the solution: ask Jev the same question (NARRATE)
# ===========================================================================
# Why: the options are data, the answer is typed, and it comes with a probability per team.
team_question = Choice(instructions="Which team should handle `message`?", criteria=TEAMS)


def ask_jev(message: str) -> dict:
    start = time.perf_counter()
    response = jev.system_one(state={"message": message}, model=JEV_MODEL,
                              questions={"team": team_question})
    ms = (time.perf_counter() - start) * 1000
    team = response.answers["team"]
    return {"team": team.choice, "probability": team.probabilities[team.choice],
            "confidence": team.confidence, "probabilities": team.probabilities, "ms": ms,
            "cost": response.usage.input_tokens * JEV_PRICE_IN}


answer = ask_jev(query)
print(f"=== JEV ({answer['ms']:.0f} ms) ===")
print(f"team: {answer['team']}   confidence: {answer['confidence']:.2f}")
top3 = sorted(answer["probabilities"].items(), key=lambda kv: -kv[1])[:3]
print("top 3:", ", ".join(f"{name} {p:.2f}" for name, p in top3))

In [ ]:
# ===========================================================================
# DEMO: the baseline to always ask for: a classifier trained on labeled data (NARRATE)
# ===========================================================================
# Why: CLINC150 ships about 15,000 labeled training messages. TF-IDF plus logistic regression
# on them is the classic answer, and it returns a probability per team too. Fair question:
# do we need Jev or an LLM at all? (scikit-learn is preinstalled in Colab.)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

clinc_train = load_dataset("clinc/clinc_oos", "plus", split="train")
train = [(row["text"], intent_to_team[intent_names[row["intent"]]])
         for row in clinc_train if intent_names[row["intent"]] != "oos"]
start = time.perf_counter()
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                      LogisticRegression(max_iter=2000))
tfidf.fit([text for text, _ in train], [team for _, team in train])
print(f"Trained on {len(train):,} labeled messages in {time.perf_counter() - start:.1f} s")


def ask_tfidf(message: str) -> dict:
    start = time.perf_counter()
    probabilities = dict(zip(tfidf.classes_, tfidf.predict_proba([message])[0]))
    ms = (time.perf_counter() - start) * 1000
    team = max(probabilities, key=probabilities.get)
    return {"team": team, "probability": probabilities[team], "ms": ms, "cost": 0.0}


answer = ask_tfidf(query)
print(f"TF-IDF: {answer['team']} {answer['probability']:.2f} ({answer['ms']:.1f} ms, no API call)")

In [ ]:
# ===========================================================================
# DEMO: 40 messages, five ways, one table (NARRATE)
# ===========================================================================
# Why: one example proves nothing. Same 40 messages, measured side by side. "format ok" counts
# replies that are exactly a team name; accuracy counts only those, as your code would.
# p50 is the median latency, p95 the slow tail (with 40 calls, the 3rd slowest).
runs = {"LLM plain": [], "LLM schema": [], "LLM logprobs": [], "Jev": [], "TF-IDF": []}
for text, label in sample:
    runs["LLM plain"].append({**ask_llm(text), "label": label})
    runs["LLM schema"].append({**ask_llm(text, schema=True), "label": label})
    runs["LLM logprobs"].append({**ask_llm_logprobs(text), "label": label})
    runs["Jev"].append({**ask_jev(text), "label": label})
    runs["TF-IDF"].append({**ask_tfidf(text), "label": label})

WHAT_YOU_GET = {"LLM plain": "label only", "LLM schema": "label only",
                "LLM logprobs": "its own tokens", "Jev": "every team", "TF-IDF": "every team"}
summary = {}
print(f"{'':12} {'format ok':>9} {'accuracy':>8} {'p50 ms':>7} {'p95 ms':>7} {'$ per 1k':>8}  probability")
for name, rows in runs.items():
    ms = sorted(r["ms"] for r in rows)
    summary[name] = {"p50": statistics.median(ms), "p95": ms[int(0.95 * (len(ms) - 1))],
                     "per_1k": 1000 * statistics.mean(r["cost"] for r in rows)}
    ok = sum(r["team"] is not None for r in rows)
    accuracy = sum(r["team"] == r["label"] for r in rows) / len(rows)
    s = summary[name]
    print(f"{name:12} {ok:>6}/40 {accuracy:>8.1%} {s['p50']:>7.0f} {s['p95']:>7.0f} "
          f"{s['per_1k']:>8.3f}  {WHAT_YOU_GET[name]}")

llm_s, jev_s = summary["LLM schema"], summary["Jev"]
print(f"\nJev vs LLM schema on this run: {llm_s['p50'] / jev_s['p50']:.1f}x faster at p50, "
      f"{llm_s['per_1k'] / jev_s['per_1k']:.0f}x cheaper per 1,000 messages.")
print("With N = 40, one message is 2.5 points of accuracy: gaps of a few points are noise.")

In [ ]:
# ===========================================================================
# DEMO: read the failures, and the messages Jev was torn on
# ===========================================================================
# Why: totals hide the story. Read what broke, and look at a sure answer next to a torn one.
print("=== LLM plain prompt: replies that are not a bare team name ===")
bad = [r["text"] for r in runs["LLM plain"] if r["team"] is None]
for text in bad:
    print(f"  {text[:70]!r}")
print(f"  ({len(bad)} of 40)")

print("\n=== JEV misses, with its confidence ===")
for (text, label), r in zip(sample, runs["Jev"]):
    if r["team"] != label:
        print(f"  {text[:40]:40} label {label:18} jev {r['team']:18} conf {r['confidence']:.2f}")

print("\n=== Jev's most and least sure answers, next to the schema LLM's label ===")
order = sorted(range(len(sample)), key=lambda i: runs["Jev"][i]["confidence"])
for i in order[-1:] + order[:2]:
    text, label = sample[i]
    j, s = runs["Jev"][i], runs["LLM schema"][i]
    top2 = sorted(j["probabilities"].items(), key=lambda kv: -kv[1])[:2]
    print(f"  {text[:40]:40} LLM: {s['team']:18} Jev: "
          + ", ".join(f"{name} {p:.2f}" for name, p in top2) + f"  (conf {j['confidence']:.2f})")
print("The LLM label looks the same in every row. Jev's numbers tell your code which row to check.")

**Figure: Results 1 of 4 - accuracy on the same 40 messages (our run, 2026-10-31). One message is 2.5 points: gaps of a few points are noise.**

<!-- DIAGRAM: p1-results-accuracy -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-results-accuracy.png" width="700" alt="Bar chart of accuracy on 40 messages: LLM plain prompt 72.5, LLM with a JSON schema 67.5, older LLM with logprobs 82.5, Jev 72.5, trained TF-IDF classifier 95">

**Figure: Results 2 of 4 - median time per message. Jev answers in about a quarter of a second; the classifier runs locally in 1 ms.**

<!-- DIAGRAM: p1-results-latency -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-results-latency.png" width="700" alt="Bar chart of median latency in milliseconds: LLM plain 739, LLM schema 879, LLM logprobs 593, Jev 235, TF-IDF 1">

**Figure: Results 3 of 4 - cost per 1,000 messages. Jev is 11 times cheaper than the schema LLM; the classifier costs nothing per call once trained.**

<!-- DIAGRAM: p1-results-cost -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-results-cost.png" width="700" alt="Bar chart of dollars per 1,000 messages: LLM plain 0.175, LLM schema 0.256, LLM logprobs 0.085, Jev 0.022, TF-IDF 0">

**Figure: Results 4 of 4 - the payoff: Jev's confidence tells your code which answers to trust. Same 40 messages, re-run 2026-10-31 (same 11 misses).**

<!-- DIAGRAM: p1-results-confidence -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-results-confidence.png" width="700" alt="Horizontal bar chart of how often Jev was right by confidence band: under 0.8, 4 of 11, 36 percent; 0.8 to 0.95, 3 of 6, 50 percent; 0.95 and up, 22 of 23, 96 percent">

> **Think About It**: Read the last chart first: above 0.95 confidence Jev was right 22 times in 23, below 0.8 only 4 times in 11. That is what lets your code route some messages and review the rest. Then the honest part: if you have 15,000 labeled messages for exactly these 10 teams, train a classifier. Jev's case is the questions you have no labels for (urgency, risk, "what kind of request is this"): one line of code each, a probability per option, about a quarter of a second. Look at Jev's misses too: some are labels a person could argue about, some are confident and wrong. What would your code do with a 0.13 confidence? With a 0.99 miss?

### 4. Lab 1: Your first Jev questions

You write Jev questions yourself: a new Choice, a function that returns a team with how sure Jev is, and an experiment on a known weak spot.

**Figure: Lab 1 map - what you build at each level; the bold boxes are your decisions.**

<!-- DIAGRAM: p1-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p1-lab-map.png" width="710" alt="Level 1: you write request_types, three descriptions that become request_question, asked about 8 messages for a type and a confidence. Level 2: classify_team(message) makes one Jev call with team_question, takes the top 2 teams and their probabilities, applies your needs_review rule, and prints 20 messages sorted by gap. Then you read the table from the smallest gap to the largest, right or wrong, and choose where the needs_review line goes. Level 3, optional: order_check(messages) asks with TEAMS forward and reversed in one call and prints the answers that changed and the accuracy of each">

#### Your Task

1. Level 1: write the options of a new `request_type` Choice
2. Level 2: write `classify_team()` and your `needs_review` rule
3. Level 3 (optional): does the order of the options change Jev's answers?

#### Level 1: Guided

Fill `request_types`: `information`, `action` and `complaint`, each with a one-line description of what that kind of message looks like. No labeled data: the descriptions are the whole question.

#### Level 2: Core

Write the body of `classify_team(message)`: one Jev call with `team_question`, returning the top team and its probability, the runner-up and its probability, the confidence, and `needs_review`. **The `needs_review` rule is your decision:** start with `False`, run the cell, read the 20 rows sorted by gap, then write your rule and justify it in a comment. Twenty messages give a first guess; Module 3 re-checks it on more data.

#### Level 3: Challenge (optional)

TypeSafe's docs warn that with long option lists Jev can lean toward the options that come first. Write `order_check(messages)`: the team question with `TEAMS` forward and reversed, in one request per message; print how many answers change and the accuracy of each order. What would you do about it in production?

#### Expected Output

- Level 1: 8 messages with a type and a confidence; with clear descriptions all 8 match
- Level 2: in our run 90% on these 20 (a random 20, easier than the demo's mix). A 0.5 gap rule sends 2 to review and the other 18 are all right
- Level 3: a few answers change with the order (1 to 5 of 40 across our runs), and the two orders can score differently

#### Hints

- Level 1: describe each option as what the customer is doing, not as a label. For a phone
  company, "upgrade" might be "the customer wants a newer phone or a bigger plan"
- Level 2: the answer's `probabilities` is a dictionary of team to probability; the top two
  teams and their gap tell you whether Jev was torn
- Level 3: two questions can share one request (Part 1's solution). Build the reversed option dictionary
  yourself

In [ ]:
# ===========================================================================
# LAB 1.1 (Guided)
# ===========================================================================
# Step 1: describe each kind of request in one line. These descriptions ARE the question.
request_types = None  # YOUR CODE: {"information": "...", "action": "...", "complaint": "..."}

LAB1_MESSAGES = [
    ("what's my checking balance", "information"),
    ("transfer $100 from checking to savings", "action"),
    ("my card got declined again and i'm really annoyed", "complaint"),
    ("book a table for two at 8 tonight", "action"),
    ("when is my credit card bill due", "information"),
    ("the app keeps logging me out, this is useless", "complaint"),
    ("cancel my hotel reservation in phoenix", "action"),
    ("how many calories are in an avocado", "information"),
]

if request_types is None:
    print("Fill request_types, then run this cell again.")
else:
    request_question = Choice(instructions="What kind of request is `message`?",
                              criteria=request_types)
    right = 0
    for text, expected in LAB1_MESSAGES:
        a = jev.system_one(state={"message": text}, model=JEV_MODEL,
                           questions={"request_type": request_question}).answers["request_type"]
        right += a.choice == expected
        print(f"{text[:48]:48} {a.choice:12} confidence {a.confidence:.2f}")
    print(f"\n{right} of {len(LAB1_MESSAGES)} match the expected type.")

In [ ]:
# ===========================================================================
# LAB 1.2 (Core)
# ===========================================================================
def classify_team(message: str) -> dict:
    """Ask Jev team_question about message in one call.

    Returns {"team", "probability", "runner_up", "runner_up_probability", "confidence",
    "needs_review"}: the top team and its probability, the second team and its probability,
    the answer's confidence, and needs_review (bool): True when Jev is too torn to route the
    message on its own. Start with needs_review = False, run the cell, read the table, then
    write your rule and justify it in a comment.
    """
    # YOUR CODE
    pass


lab_sample = [(text, team) for text, team in random.Random(7).sample(in_scope, 20)]
results = []
for text, label in lab_sample:
    result = classify_team(text)
    if result is None:
        print("classify_team() returned None: write its body above.")
        break
    results.append((text, label, result))

if results:
    accuracy = sum(r["team"] == label for _, label, r in results) / len(results)
    auto = [(label, r) for _, label, r in results if not r["needs_review"]]
    auto_accuracy = sum(r["team"] == label for label, r in auto) / len(auto) if auto else 0.0
    print(f"Accuracy on {len(results)} messages: {accuracy:.0%}")
    print(f"Sent to review: {len(results) - len(auto)}; accuracy of the rest: {auto_accuracy:.0%}\n")
    print(f"{'top team':18} {'p':>4}  {'runner-up':18} {'p':>4} {'gap':>5} {'conf':>5}  right  review | message")
    for text, label, r in sorted(results, key=lambda x: x[2]["probability"] - x[2]["runner_up_probability"]):
        gap = r["probability"] - r["runner_up_probability"]
        print(f"{r['team']:18} {r['probability']:4.2f}  {r['runner_up']:18} {r['runner_up_probability']:4.2f} "
              f"{gap:5.2f} {r['confidence']:5.2f}  {'yes' if r['team'] == label else 'NO ':5}  "
              f"{str(r['needs_review']):6} | {text[:34]}")

In [ ]:
# ===========================================================================
# LAB 1.3 (Challenge, optional)
# ===========================================================================
# Spec: order_check(messages) asks the team question twice per message in ONE request:
#   "forward" with TEAMS in its normal order, "reversed" with the options reversed.
# It prints how many answers changed, and the accuracy of each order against the labels.
# Then answer in a comment: what would you do about it in production?


def order_check(messages: list[tuple[str, str]]) -> None:
    # YOUR CODE
    pass


order_check(random.Random(11).sample(in_scope, 40))

## Part 2: One request, many judgments

Real triage needs more than a team: **how urgent** is it, and is the customer's **money or access at risk**? Those decide whether a person picks it up now.

### 1. The problem

The shortcut is one question that makes the whole decision: *"Should this message be escalated to a human?"* A lost card that must be frozen now scores 0.60. Urgent? Risky? The number cannot say, and no cut-off expresses "money at risk always goes to a person".

**Figure: The problem - one "escalate?" Noul on the lost card: 0.60, the rule hidden inside the model, and any cut-off you pick is a guess (red).**

<!-- DIAGRAM: p2-problem-hidden-rule -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-problem-hidden-rule.png" width="750" alt="Top row: the message i lost my card at the mall, need it frozen right away, goes to one Noul, escalate, which hides three judgments, urgent, risky and which team; Jev answers 0.60. Bottom row: your code can only compare escalate with a cut-off you guess, 0.5 or 0.7; at 0.7 a card that must be frozen is not escalated, and a new policy means rewriting the question">

### 2. The solution

Split the decision into the **judgments** it depends on, ask them in **one request**, and write the **decision in your own code**: `team` (Choice), `urgency` (Score: 0 can wait, 2 now) and `risk` (Noul: money or access at risk?). A risk between 0.3 and 0.7 means the message does not settle it, so that band gets its own action: REVIEW, a person looks.

**The rule: Jev judges, your code decides.** Ask Jev what is true, never what to do.

**Figure: The solution - same lost card, same layout: three judgments in one request, then `triage()` checks the rules in order and escalates because risk is 0.97.**

<!-- DIAGRAM: p2-solution-judgments-rules -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-solution-judgments-rules.png" width="680" alt="Top row: the lost card message goes to Jev with three questions in one request, team as a Choice, urgency as a Score, risk as a Noul; Jev answers credit_cards, urgency 2.00, risk 0.97. Bottom row: triage checks the rules in order, risk at least 0.7 ESCALATE yes, then risk above 0.3 REVIEW, urgency at least 1.5 PRIORITY, else ROUTE; the outcome is ESCALATE to a person now because risk is 0.97, and a new policy means changing a threshold">

### 3. Demo

First the problem live: the escalate shortcut on six messages. Then the solution with different judgments (frustration and travel plans), so the lab stays yours: one call with three question types, the tokens it saves, every field of the answer, a rule in code, and which model version answered.

In [ ]:
# ===========================================================================
# DEMO: the problem: one question that makes the whole decision (NARRATE)
# ===========================================================================
# Why: the number comes back, but nothing tells you WHY, or how to change the policy.
escalate_question = Noul(instructions="Should `message` be escalated to a human?")
escalate = {}
for text in ["i lost my card at the mall and need it frozen right away",
             "there's a charge on my statement i never made",
             "why was my card declined at the gas station",
             "the bank app won't let me log in",
             "i need to book a flight to boston next month",
             "what's my checking balance"]:
    r = jev.system_one(state={"message": text}, model=JEV_MODEL,
                       questions={"escalate": escalate_question})
    escalate[text] = r.answers["escalate"].noul
    print(f"{text[:50]:50} escalate = {escalate[text]:.2f}")
lost = escalate["i lost my card at the mall and need it frozen right away"]
print(f"\nThe lost card that must be frozen now scores {lost:.2f}. Urgent? Risky? Both? "
      "The number cannot say, and no threshold expresses 'money at risk always goes to a person'.")

**Figure: The problem, measured - one escalate number per message. Which one is risky, which only urgent? The bars cannot say.**

<!-- DIAGRAM: p2-demo-escalate -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-demo-escalate.png" width="700" alt="Horizontal bar chart of the escalate Noul for six messages: lost card 0.60, unknown charge 0.92, declined card 0.41, app login 0.63, flight next month 0.21, balance 0.19">

In [ ]:
# ===========================================================================
# DEMO: the solution: one state, three question types, one call (NARRATE)
# ===========================================================================
# Why: questions that share a state go in one request; the state is read, and paid for,
# once. Here: a Choice, a Score and a Noul about the same message. Then the same three
# questions as three separate calls, so you can see the input tokens you would pay for.
frustration_question = Score(
    instructions="How frustrated does the customer sound in `message`?",
    criteria=["the customer sounds calm",
              "the customer sounds annoyed",
              "the customer sounds angry or desperate"])
travel_question = Noul(instructions="Does `message` mention travel plans?")
travel_message = "my flight to rome is tomorrow and my card keeps getting declined, what do i do?"
demo_questions = {"team": team_question, "frustration": frustration_question,
                  "travel_plans": travel_question}

start = time.perf_counter()
response = jev.system_one(state={"message": travel_message}, model=JEV_MODEL,
                          questions=demo_questions)
print(f"One call, three answers: {(time.perf_counter() - start) * 1000:.0f} ms, "
      f"{response.usage.input_tokens} input tokens, answered by {response.model}")
for name, answer in response.answers.items():
    print(f"  {name:13} {answer.type}")

separate = [jev.system_one(state={"message": travel_message}, model=JEV_MODEL,
                           questions={name: question}).usage.input_tokens
            for name, question in demo_questions.items()]
bundled = response.usage.input_tokens
print(f"\nInput tokens: bundled {bundled}, three separate calls {sum(separate)} ({separate})")
print(f"Cost per 1,000 messages: bundled ${1000 * bundled * JEV_PRICE_IN:.4f}, "
      f"separate ${1000 * sum(separate) * JEV_PRICE_IN:.4f}")

**Figure: Bundling, measured - three questions in one call read the message once: 611 input tokens instead of 1,179, about half the cost.**

<!-- DIAGRAM: p2-demo-tokens -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-demo-tokens.png" width="700" alt="Bar chart of input tokens for three questions about one message: one bundled call 611, three separate calls 1,179">

In [ ]:
# ===========================================================================
# DEMO: read every field: choice, score, noul, probabilities, confidence (NARRATE)
# ===========================================================================
# Why: probabilities say how likely each option is; confidence says how much of the
# probability sits on one option. A Score can land in the middle because Jev is split.
team = response.answers["team"]
frustration = response.answers["frustration"]
travel = response.answers["travel_plans"]

top3 = sorted(team.probabilities.items(), key=lambda kv: -kv[1])[:3]
print(f"team.choice       = {team.choice}   confidence {team.confidence:.2f}")
print("   top 3:", ", ".join(f"{name} {p:.2f}" for name, p in top3))
print(f"frustration.score = {frustration.score:.2f} on 0..2   confidence {frustration.confidence:.2f}")
for level, p in frustration.probabilities.items():
    print(f"   level {level}: {p:.2f}  {frustration.legend[level]}")
print(f"travel_plans.noul = {travel.noul:.2f}   (one number: the probability the statement is true)")

In [ ]:
# ===========================================================================
# DEMO: Jev judges, your code decides (NARRATE)
# ===========================================================================
# Why: the decision is plain Python over the answers. Change the policy, change a line.
# We run it on the travel message, then on the six messages from the problem above.
def demo_action(answers: dict) -> str:
    if answers["travel_plans"].noul >= 0.7:
        return "REMIND: set a travel notice"
    if answers["frustration"].score >= 1.5:
        return "CALL BACK: an upset customer"
    return "ROUTE to " + answers["team"].choice


print(f"message: {travel_message}")
print(f"action:  {demo_action(response.answers)}\n")

print(f"{'message':44} {'escalate':>8}  {'frust':>5} {'travel':>6}  action")
for text, escalate_score in escalate.items():
    a = jev.system_one(state={"message": text}, model=JEV_MODEL, questions=demo_questions).answers
    print(f"{text[:44]:44} {escalate_score:>8.2f}  {a['frustration'].score:>5.2f} "
          f"{a['travel_plans'].noul:>6.2f}  {demo_action(a)}")
print("\nEvery action now has a reason you can read in its row. Look at the lost card: these rules "
      "have no judgment about money at risk, so they cannot escalate it. The fix is one more "
      "judgment and one more rule, and that is your lab.")

**Figure: The solution, measured - the same six messages: separate judgments and a readable rule pick each action. The orange rows are the gap your lab closes: no risk rule yet.**

<!-- DIAGRAM: p2-demo-actions -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-demo-actions.png" width="420" alt="Rules in order: travel_plans at least 0.7 REMIND, frustration at least 1.5 CALL BACK, else ROUTE. Six rows: lost card, escalate was 0.60, frustration 1.61, CALL BACK, no risk rule yet; unknown charge, was 0.92, frustration 0.42, ROUTE credit_cards, no risk rule yet; declined card, was 0.41, ROUTE credit_cards; app login, was 0.63, ROUTE banking; flight next month, was 0.21, travel 0.99, REMIND; balance, was 0.19, ROUTE banking">

In [ ]:
# ===========================================================================
# DEMO: pinned versions: which model answered you? (NARRATE)
# ===========================================================================
# Why: "jev-latest" moves when TypeSafe ships a release; a pinned id does not. Every
# response says which version answered, so log it next to the decision.
for model_id in ("jev-latest", JEV_MODEL):
    r = jev.system_one(state={"message": travel_message}, model=model_id,
                       questions={"travel_plans": travel_question})
    print(f"asked for {model_id:11} -> answered by {r.model:11}  travel_plans = {r.answers['travel_plans'].noul:.2f}")

> **Think About It**: In our run the travel message's frustration score came back at 1.42,
> split between "annoyed" (0.42) and "angry" (0.50), with confidence 0.13. Would you act on
> the score alone, or on the level probabilities? And in the "Jev judges, your code decides"
> cell, the rule for "REMIND" uses 0.7: when `jev-latest` moves to the next release, which of
> your thresholds would you re-check before you follow it?

### 4. Lab 2: Triage in one call

This is the first real version of our triage layer: one Jev call per message returns the team,
the urgency and a risk signal, and your code turns those judgments into an action.

**Figure: Lab 2 map - what you build at each level; the bold boxes are yours to write.**

<!-- DIAGRAM: p2-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_01/p2-lab-map.png" width="650" alt="Level 1: you write urgency_levels, three levels about time, which become urgency_question, a Score from 0 to 2, and risk_instructions, money or access at risk, which become risk_question, a Noul from 0 to 1. Level 2: triage(message) makes one Jev call for team, urgency and risk, then checks in order risk at least 0.7 ESCALATE, risk above 0.3 REVIEW, urgency at least 1.5 PRIORITY, else ROUTE. Expected on 12 messages in our run: 3 ESCALATE with risk 0.81 to 0.97, 2 REVIEW with risk 0.39 and 0.49, 1 PRIORITY, the declined card, 6 ROUTE with risk 0.01. Level 3, optional: compare_calls(messages) times bundled, sequential and parallel calls and prints p50 ms and input tokens for each">

#### Your Task

1. Write the urgency levels and the risk question (Level 1)
2. Write `triage()`: one Jev call, then the rules below, in order (Level 2)
3. Optional: find out what asking all questions in one call really buys you (Level 3)

#### Level 1: Guided

Fill `urgency_levels` (three situations, least to most urgent, about **time**) and
`risk_instructions` (one statement Jev can judge true or false: the customer's **money or
account access** is at risk). Keep urgency and risk apart: they are different judgments. The
cell runs both questions on three messages. Stuck? The FALLBACK cell below fills them in so you
can move on, but it holds a working answer: do not read it until you have tried.

#### Level 2: Core

Write the body of `triage(message)`: one `jev.system_one` call with `team_question`,
`urgency_question` and `risk_question`, then this policy, checked in order: the four rules in the Level 2 row of the map above (also in the docstring).

The thresholds are given: 0.7 and 0.3 are TypeSafe's suggested starting points for a Noul,
and 1.5 sits halfway between "next few days" and "within the hour". Run the 12 messages and
explain, in a comment, any row you disagree with. Choosing thresholds from data is Module 3.

#### Level 3: Challenge (optional)

Write `compare_calls(messages)`: answer the three questions for each message in three ways:
one bundled call, three calls one after another, and three calls in parallel threads. Print
p50 latency and mean input tokens for each. Then answer in a comment: what does bundling buy
you, and when would you split the calls anyway?

#### Expected Output

- Level 1: the lost card scores high on urgency and above 0.7 on risk; the weather question
  scores near 0 on both
- Level 2: a 12-row table. In our run, with the solution's wording and thresholds (0.7, 0.3,
  1.5): 3 ESCALATE (lost card, unknown charge, someone used my card), 2 REVIEW (lost wallet,
  app login), 1 PRIORITY (declined card), 6 ROUTE. Your wording can move a row
- Level 3: three rows of numbers and your answer in a comment

#### Hints

- Level 1: describe each Score level as a situation a person would recognize, not "more
  urgent than the previous level"
- Level 2: `response.answers["risk"].noul` is a float from 0 to 1;
  `response.answers["urgency"].score` is a float from 0 to 2. TypeSafe suggests 0.7 and 0.3
  as starting thresholds for a Noul: above 0.7 reads as yes, below 0.3 as no
- Level 3: `time.perf_counter()`, `response.usage.input_tokens`, `statistics.median` and
  `concurrent.futures.ThreadPoolExecutor`. Shuffle the order of the three ways per message

In [ ]:
# ===========================================================================
# LAB 2.1 (Guided)
# ===========================================================================
# Step 1: describe three urgency levels, least to most urgent, each as a situation in time.
urgency_levels = None  # YOUR CODE: a list of 3 strings

# Step 2: one statement Jev can judge true or false: money or account access is at risk.
risk_instructions = None  # YOUR CODE: a string that refers to `message`

if urgency_levels is None or risk_instructions is None:
    print("Fill urgency_levels and risk_instructions, then run this cell again.")
else:
    urgency_question = Score(instructions="How urgent is `message` for the customer?",
                             criteria=urgency_levels)
    risk_question = Noul(instructions=risk_instructions)
    for text in ["i lost my card at the mall and need it frozen right away",
                 "there's a charge on my statement i never made",
                 "will it rain tomorrow"]:
        r = jev.system_one(state={"message": text}, model=JEV_MODEL,
                           questions={"urgency": urgency_question, "risk": risk_question})
        print(f"{text[:50]:50} urgency {r.answers['urgency'].score:.2f}   risk {r.answers['risk'].noul:.2f}")

In [ ]:
# ===========================================================================
# FALLBACK: Ensures Level 2 works even if you didn't complete Level 1
# ===========================================================================
if urgency_levels is None or risk_instructions is None:
    print("Using working urgency and risk questions so Level 2 still runs.")
    urgency_levels = [
        "the customer is asking for information and nothing needs doing soon",
        "the customer needs something done in the next few days",
        "the customer needs something done within the hour",
    ]
    risk_instructions = ("Does `message` report a lost or stolen card, a charge the customer "
                         "did not make, or a locked account?")
    urgency_question = Score(instructions="How urgent is `message` for the customer?",
                             criteria=urgency_levels)
    risk_question = Noul(instructions=risk_instructions)

In [ ]:
# ===========================================================================
# LAB 2.2 (Core)
# ===========================================================================
LAB2_MESSAGES = [
    "i lost my card at the mall and need it frozen right away",
    "there's a charge on my statement i never made",
    "i think someone used my card but i'm not sure",
    "i can't find my wallet, maybe it's in the car",
    "the bank app won't let me log in",
    "why was my card declined at the gas station",
    "what's my checking balance",
    "how do i make lasagna",
    "will it rain tomorrow",
    "tell me a joke",
    "i need to book a flight to boston next month",
    "what movies are playing",
]


def triage(message: str) -> dict:
    """One Jev call with team_question, urgency_question and risk_question.

    Returns {"team", "urgency", "risk", "action"}. The policy, in words, checked in order:
    1. risk >= 0.7 (money or access clearly at risk): "ESCALATE" to a person now
    2. risk > 0.3 (Jev not settled on the risk): "REVIEW" by a person
    3. urgency >= 1.5 (urgent): "PRIORITY" to the team
    4. anything else: "ROUTE" to the team
    """
    # YOUR CODE
    pass


print(f"{'message':48} {'team':18} {'urg':>5} {'risk':>5}  action")
for text in LAB2_MESSAGES:
    result = triage(text)
    if result is None:
        print("triage() returned None: write its body above.")
        break
    print(f"{text[:48]:48} {result['team']:18} {result['urgency']:>5.2f} {result['risk']:>5.2f}  {result['action']}")

In [ ]:
# ===========================================================================
# LAB 2.3 (Challenge, optional)
# ===========================================================================
# Spec: compare_calls(messages) answers team, urgency and risk for every message three ways:
#   A) one call with all three questions
#   B) three calls, one question each, one after another
#   C) three calls, one question each, in parallel threads
# and prints, for A, B and C, the p50 latency in ms and the mean input tokens per message.
# Then, in a comment below the call: what does bundling buy you, and when would you split?


def compare_calls(messages: list[str]) -> None:
    # YOUR CODE
    pass


compare_calls(LAB2_MESSAGES[:8])

# Before production

What this notebook does not settle, and where each answer lives. Numbers are from our run on
2026-10-31.

| Question | Where we stand |
|---|---|
| Cost of `triage()` | About 630 input tokens per message, so about $0.027 per 1,000 messages at $0.042 per million input tokens (docs.typesafe.ai/models). Output is free |
| Latency | Single calls ran about 235 ms at p50 and about 270 ms at p95 (Part 1 table, N = 40). Measure p95 on your own traffic before you promise anything |
| Rate limits | Set per organization, in requests and tokens **per second**; over the limit you get a 429 (docs.typesafe.ai/models). A class of 25 on one key can hit it |
| Retries and timeouts | The CONFIG cell sets them: Jev retries 408, 429 and 5xx twice inside a 30 s budget, then raises (docs.typesafe.ai/sdk/python/api/retries) |
| When a call fails | Treat it as an error, never as an answer: a failed risk call is not "no risk". Route the message to the human queue |
| Thresholds | Ours rest on 12 to 20 messages. Module 3 measures them on hundreds |
| A trained classifier | If you have labeled data for a fixed set of classes, keep it as the baseline: it won Part 1's table |

# Summary

## What you can now do

1. **Say what Jev is.** A System One model: it reads a state and answers typed questions
   (Choice, Score, Noul) with probabilities. It never writes text.
2. **Choose it for the right reason.** In our run (2026-10-31, 40 messages) Jev was about as
   accurate as an LLM with a JSON schema, 3.7 times faster at p50 and 11 times cheaper per
   1,000 messages, and it gave a probability for every team. A classifier trained on 15,000
   labeled messages beat both on accuracy. Jev earns its place where you have no labels: a
   new judgment is one line of code. Your table printed your own ratios.
3. **Keep the decision in code.** Jev judges team, urgency and risk; `triage()` decides, where
   you can read it, test it and change a threshold.

## Looking Ahead

Module 2, Designing Good Questions. Our risk question joins three conditions with "or", so
when it fires we cannot tell which risk it was, and a question about dates is a trap Jev reads
as text. We split fuzzy judgments into literal ones and move logic and dates into code.

## Homework

Write 10 messages like the ones your own support inbox gets (no real customer data) and run
`triage()` on them. Which rule fires most? Pick one answer you disagree with: is the fix in the
state, the question, or the code?